In [2]:
# =============================================================================
# XGBOOST WITH OPTUNA - CONSERVATIVE (Mimicking Your 0.798 GB Model)
# Expected: CV ~0.88-0.90, Kaggle 0.82-0.85
# Runtime: ~1.5-2 hours    on dataset version 6 (ta tips guilem nieuwe)   29/11 14:00 
# =============================================================================

import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score
from datetime import datetime

print("="*80)
print("XGBOOST + OPTUNA - CONSERVATIVE TUNING")
print("="*80)

XGBOOST + OPTUNA - CONSERVATIVE TUNING


In [3]:
# =============================================================================
# 2. DEFINE EXPERIMENT PARAMETERS
# =============================================================================
EXPERIMENT_NAME = "xgboost_optuna"
MODEL_DESCRIPTION = "XGBoost with Bayesian Hyperparameter Optimization on preprocessed 6 data"

In [4]:


# =============================================================================
# 1. LOAD DATA
# =============================================================================

print("Loading preprocessed data...")
X = pd.read_pickle('../../data/processed/X_train_processed.pkl')
y = pd.read_pickle('../../data/processed/y_train.pkl')
X_test = pd.read_pickle('../../data/processed/X_test_processed.pkl')
test_ids = pd.read_pickle('../../data/processed/test_ids.pkl')

print(f"X_train: {X.shape}")
print(f"y_train: {y.shape}")

scale_pos_weight = (y == 0).sum() / (y == 1).sum()
print(f"Scale pos weight: {scale_pos_weight:.2f}")

# =============================================================================
# 2. OPTUNA OBJECTIVE FUNCTION - CONSERVATIVE PARAMS
# =============================================================================

def objective(trial):
    """
    Conservative parameter search - mimicking your 0.798 GB model
    Focus: Prevent overfitting, maximize generalization
    """
    
    params = {
        # Number of trees - conservative range
        'n_estimators': trial.suggest_int('n_estimators', 100, 400),
        
        # Learning rate - lower is safer
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        
        # Tree depth - SHALLOW trees (like your GB)
        'max_depth': trial.suggest_int('max_depth', 2, 5),
        
        # Minimum samples - HIGH values (like GB's min_samples_leaf)
        # min_child_weight in XGBoost ≈ min_samples_leaf in sklearn
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 30),
        
        # Regularization - STRONG
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),  # L1
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 10.0),  # L2
        'gamma': trial.suggest_float('gamma', 0.0, 1.0),  # Min loss reduction
        
        # Sampling - AGGRESSIVE (like GB's subsample)
        'subsample': trial.suggest_float('subsample', 0.5, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.5, 0.9),
        
        # Fixed params
        'scale_pos_weight': scale_pos_weight,
        'random_state': 42,
        'tree_method': 'hist',
        'eval_metric': 'auc',
        'verbosity': 0
    }
    
    # 5-fold CV (same as your GB model)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    model = xgb.XGBClassifier(**params)
    
    # Cross-validation score
    cv_scores = cross_val_score(
        model, X, y,
        cv=cv,
        scoring='roc_auc',
        n_jobs=-1
    )
    
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()
    
    # IMPORTANT: Also penalize high variance across folds
    # This helps prevent overfitting
    score = cv_mean - 0.5 * cv_std
    
    return score

# =============================================================================
# 3. RUN OPTUNA OPTIMIZATION
# =============================================================================

print("\n" + "="*80)
print("STARTING OPTUNA OPTIMIZATION")
print("="*80)
print("\nSettings:")
print("  Trials: 100")
print("  CV folds: 5")
print("  Focus: Conservative params (prevent overfitting)")
print("  Expected runtime: ~1.5-2 hours")
print("\nProgress will be shown every 10 trials...")

# Create study
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

# Optimize
study.optimize(
    objective,
    n_trials=100,
    show_progress_bar=True,
    n_jobs=1  # XGBoost already uses all cores
)

# =============================================================================
# 4. RESULTS
# =============================================================================

print("\n" + "="*80)
print("OPTUNA RESULTS")
print("="*80)

print(f"\nBest trial:")
print(f"  Score (CV - 0.5*std): {study.best_value:.4f}")
print(f"\nBest parameters:")

best_params = study.best_params
for param, value in best_params.items():
    print(f"  {param}: {value}")

# =============================================================================
# 5. TRAIN FINAL MODEL WITH BEST PARAMS
# =============================================================================

print("\n" + "="*80)
print("TRAINING FINAL MODEL")
print("="*80)

# Add fixed params
final_params = best_params.copy()
final_params.update({
    'scale_pos_weight': scale_pos_weight,
    'random_state': 42,
    'tree_method': 'hist',
    'eval_metric': 'auc'
})

# Train with best params
final_model = xgb.XGBClassifier(**final_params)

# Get CV score with best params
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(final_model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f"\nFinal CV Performance:")
print(f"  CV scores: {[f'{s:.4f}' for s in cv_scores]}")
print(f"  Mean: {cv_scores.mean():.4f}")
print(f"  Std: {cv_scores.std():.4f}")

# Train on full data
final_model.fit(X, y)

# Training performance
train_proba = final_model.predict_proba(X)[:, 1]
train_auc = roc_auc_score(y, train_proba)

print(f"\nOverfitting Analysis:")
print(f"  Training AUC: {train_auc:.4f}")
print(f"  CV AUC: {cv_scores.mean():.4f}")
print(f"  Gap: {train_auc - cv_scores.mean():.4f}")

if train_auc - cv_scores.mean() < 0.05:
    print("  ✓ Excellent! Minimal overfitting")
elif train_auc - cv_scores.mean() < 0.10:
    print("  ✓ Good! Acceptable overfitting")
else:
    print("  ⚠️ Moderate overfitting")

# =============================================================================
# 6. GENERATE PREDICTIONS
# =============================================================================

print("\n--- Generating test predictions ---")

test_proba = final_model.predict_proba(X_test)[:, 1]

print(f"\nTest prediction statistics:")
print(f"  Min: {test_proba.min():.4f}")
print(f"  Max: {test_proba.max():.4f}")
print(f"  Mean: {test_proba.mean():.4f}")
print(f"  Median: {np.median(test_proba):.4f}")

[I 2025-11-29 14:03:35,277] A new study created in memory with name: no-name-301061ed-61ff-4e8f-9041-8910cba56c37


Loading preprocessed data...
X_train: (20885, 95)
y_train: (20885,)
Scale pos weight: 7.91

STARTING OPTUNA OPTIMIZATION

Settings:
  Trials: 100
  CV folds: 5
  Focus: Conservative params (prevent overfitting)
  Expected runtime: ~1.5-2 hours

Progress will be shown every 10 trials...


Best trial: 0. Best value: 0.90231:   1%|          | 1/100 [00:03<05:31,  3.35s/it]

[I 2025-11-29 14:03:38,622] Trial 0 finished with value: 0.9023102847994133 and parameters: {'n_estimators': 212, 'learning_rate': 0.08927180304353628, 'max_depth': 4, 'min_child_weight': 20, 'reg_alpha': 0.7800932022121826, 'reg_lambda': 2.403950683025824, 'gamma': 0.05808361216819946, 'subsample': 0.846470458309974, 'colsample_bytree': 0.7404460046972835, 'colsample_bylevel': 0.7832290311184182}. Best is trial 0 with value: 0.9023102847994133.


Best trial: 0. Best value: 0.90231:   2%|▏         | 2/100 [00:05<04:31,  2.77s/it]

...

[I 2025-11-29 14:09:30,105] Trial 97 finished with value: 0.902025260405824 and parameters: {'n_estimators': 310, 'learning_rate': 0.08207462780742304, 'max_depth': 5, 'min_child_weight': 25, 'reg_alpha': 1.1916224201706413, 'reg_lambda': 1.6413267294648974, 'gamma': 0.39411453701238147, 'subsample': 0.8554307451587321, 'colsample_bytree': 0.518748465157474, 'colsample_bylevel': 0.6962156649048652}. Best is trial 91 with value: 0.9027562413669349.


Best trial: 91. Best value: 0.902756:  99%|█████████▉| 99/100 [05:59<00:03,  3.44s/it]

[I 2025-11-29 14:09:34,357] Trial 98 finished with value: 0.8945255347901635 and parameters: {'n_estimators': 332, 'learning_rate': 0.06941240257436077, 'max_depth': 2, 'min_child_weight': 20, 'reg_alpha': 1.411272783649769, 'reg_lambda': 3.088860353609993, 'gamma': 0.3240144967016656, 'subsample': 0.8811943826616715, 'colsample_bytree': 0.5985109344020747, 'colsample_bylevel': 0.843706456232382}. Best is trial 91 with value: 0.9027562413669349.


Best trial: 91. Best value: 0.902756: 100%|██████████| 100/100 [06:01<00:00,  3.62s/it]


[I 2025-11-29 14:09:36,924] Trial 99 finished with value: 0.9018931335561489 and parameters: {'n_estimators': 325, 'learning_rate': 0.0538215383508876, 'max_depth': 5, 'min_child_weight': 23, 'reg_alpha': 3.6681658794303367, 'reg_lambda': 2.250487136467343, 'gamma': 0.19048635322039092, 'subsample': 0.8238377655262669, 'colsample_bytree': 0.769224770285927, 'colsample_bylevel': 0.6121260807735086}. Best is trial 91 with value: 0.9027562413669349.

OPTUNA RESULTS

Best trial:
  Score (CV - 0.5*std): 0.9028

Best parameters:
  n_estimators: 311
  learning_rate: 0.0652513568231865
  max_depth: 5
  min_child_weight: 23
  reg_alpha: 0.6691354183070404
  reg_lambda: 2.8697079710941367
  gamma: 0.25886168393622105
  subsample: 0.856204786349967
  colsample_bytree: 0.7295944788657113
  colsample_bylevel: 0.6396156863414592

TRAINING FINAL MODEL

Final CV Performance:
  CV scores: ['0.8982', '0.9054', '0.9075', '0.9191', '0.9015']
  Mean: 0.9063
  Std: 0.0072

Overfitting Analysis:
  Training A

In [5]:
# =============================================================================
# 7. CREATE SUBMISSION FILE
# =============================================================================
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
submission_filename = f"../../outputs/predictions/XG_boost/{EXPERIMENT_NAME}_{timestamp}.csv"

submission = pd.DataFrame({
    'icustay_id': test_ids,
    'prediction': test_proba
})
submission.to_csv(submission_filename, index=False)
print(f"\n✓ Submission saved: {submission_filename}")


✓ Submission saved: ../../outputs/predictions/XG_boost/xgboost_optuna_20251129_1409.csv


In [6]:
# =============================================================================
# 8. FEATURE IMPORTANCE
# =============================================================================

print("\n" + "="*80)
print("TOP 20 FEATURES")
print("="*80)

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n" + feature_importance.head(20).to_string(index=False))

# =============================================================================
# 9. COMPARISON & PREDICTION
# =============================================================================

print("\n" + "="*80)
print("PERFORMANCE COMPARISON")
print("="*80)

print(f"\nYour previous models:")
print(f"  Gradient Boosting: CV ??? → Kaggle 0.798")
print(f"  XGBoost baseline:  CV 0.903 → Kaggle 0.789")

print(f"\nThis model (XGBoost + Optuna):")
print(f"  CV: {cv_scores.mean():.4f}")
print(f"  Expected Kaggle: ~{cv_scores.mean() - 0.09:.3f} - {cv_scores.mean() - 0.05:.3f}")

print(f"\n{'='*80}")
print("NEXT STEPS:")
print(f"{'='*80}")
print(f"\n1. Upload {submission_file} to Kaggle")
print(f"2. Compare to your 0.798 best")
print(f"3. If this beats it → we're on the right track!")
print(f"4. If not → try ensemble (XGBoost + GB + RF)")


TOP 20 FEATURES

                 feature  importance
 has_respiratory_failure    0.082422
            ICD9_encoded    0.060517
          Severity_Score    0.047272
    primary_diag_encoded    0.042219
                 has_aki    0.039112
              has_sepsis    0.033520
ADMISSION_TYPE_EMERGENCY    0.026085
                SpO2_Min    0.021595
               SysBP_Min    0.020341
            Hypoglycemia    0.019632
    n_previous_icu_stays    0.019402
   MARITAL_STATUS_SINGLE    0.016868
             n_diagnoses    0.016199
       ETHNICITY_UNKNOWN    0.015714
       is_frequent_flyer    0.014730
              TempC_Mean    0.013504
           has_pneumonia    0.013494
              MeanBP_Min    0.012154
               SpO2_Mean    0.011799
      INSURANCE_Medicare    0.011399

PERFORMANCE COMPARISON

Your previous models:
  Gradient Boosting: CV ??? → Kaggle 0.798
  XGBoost baseline:  CV 0.903 → Kaggle 0.789

This model (XGBoost + Optuna):
  CV: 0.9063
  Expected Kaggle: ~0.816

NameError: name 'submission_file' is not defined

In [7]:






# =============================================================================
# 10. SAVE MODEL & STUDY
# =============================================================================

import pickle

# Save model
model_file = f"../../outputs/models/xgboost_optuna_conservative_{timestamp}.pkl"
with open(model_file, 'wb') as f:
    pickle.dump(final_model, f)
print(f"\n✓ Model saved: {model_file}")

# Save optuna study
study_file = f"../../outputs/models/optuna_study_{timestamp}.pkl"
with open(study_file, 'wb') as f:
    pickle.dump(study, f)
print(f"✓ Study saved: {study_file}")

print("\n" + "="*80)
print("OPTIMIZATION COMPLETE!")
print("="*80)


✓ Model saved: ../../outputs/models/xgboost_optuna_conservative_20251129_1409.pkl
✓ Study saved: ../../outputs/models/optuna_study_20251129_1409.pkl

OPTIMIZATION COMPLETE!
